In [94]:
# Setup
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("C:/Users/hlarbi/risk_management", os.getcwd())
print(os.listdir())

C:/Users/hlarbi/risk_management c:\Users\hlarbi\risk_management\src
['check_data.py', 'download_data.py', 'RM_responses.ipynb']


## Q1) Stock selection & daily market data

We select ASML as the underlying asset.  
We load the daily market data from our dataset (`ASML_data.csv`). The valuation date is the last available observation.

In [95]:
import pandas as pd

ASML_CSV = "../data/ASML_data.csv"

asml = pd.read_csv(ASML_CSV, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

valuation_date = asml["Date"].iloc[-1]
S0 = float(asml["Close"].iloc[-1])

print("Stock: ASML")
print("Rows:", len(asml))
print("Valuation date:", valuation_date.date())
print("Spot price S0 (Close):", S0)

Stock: ASML
Rows: 1761
Valuation date: 2024-12-31
Spot price S0 (Close): 685.6155395507812


## Q2) Pricing inputs: risk-free rate, maturity and volatility
### (a) Risk-free rate \(r\)
To discount the strike in Black–Scholes we need an annualized “risk-free” rate in EUR.  
The euro area’s risk-free benchmark rate recommended by the ECB-convened Working Group on euro risk-free rates is the €STR (Euro Short-Term Rate). €STR is designed to be a near risk-free reference rate for EUR markets (European Central Bank, 2023).

Because our option maturity is 1 year, we need a 1-year rate. We therefore take the ECB euro area yield curve spot rate at 1-year maturity.
Series key: YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_1Y. This series is a government bond yield curve for the euro area using AAA-rated issuers and is published by the ECB with the Svensson model (continuous compounding) (European Central Bank, 2026).

Date matching: our valuation date is 2024-12-31 (last ASML observation). If the ECB curve is not available on that date we use the closest previous business day.   In our dataset the closest ECB curve date is 2024-12-30, giving:

 r = 0.02178646
 
### (b) Maturity \(T\)
We choose a projection / option maturity of T = 1 year
This is consistent with using a 1-year risk-free rate from the ECB curve.

### (c) Annualized volatility ($\sigma$)
We estimate historical volatility using log returns over a window length equal to T.
Let $S_t$ be the daily closing price. Daily log return:

$$
r_t = \ln\left(\frac{S_t}{S_{t-1}}\right)
$$

Annualized volatility:

$$
\sigma = \mathrm{std}(r_t)\sqrt{252}
$$

Using the last 252 trading days (since $T=1$), we obtain:

$$
\sigma = 0.457381
$$


### References
1. European Central Bank (2023) Working Group on euro risk-free rates, European Central Bank. Available at: https://www.ecb.europa.eu/stats/euro-short-term-rates/interest_rate_benchmarks/WG_euro_risk-free_rates/html/index.en.html (Accessed: 02 March 2026).
2. Yield curve spot rate, 1-year maturity - government bond, nominal, all issuers whose rating is triple A - euro area (changing composition), Euro Area, daily - businessweek (2026) YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_1Y | ECB Data Portal. Available at: https://data.ecb.europa.eu/data/datasets/YC/YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_1Y?chart_props=W3sibm9kZUlkIjoiMTg5NTQ4NiIsInByb3BlcnRpZXMiOlt7ImNvbG9ySGV4IjoiIiwiY29sb3JUeXBlIjoiIiwiY2hhcnRUeXBlIjoibGluZWNoYXJ0IiwibGluZVN0eWxlIjoiU29saWQiLCJsaW5lV2lkdGgiOiIxLjUiLCJheGlzUG9zaXRpb24iOiJsZWZ0Iiwib2JzZXJ2YXRpb25WYWx1ZSI6ZmFsc2UsImRhdGVzIjpbIjIwMjMtMTItMzBUMjM6MDA6MDAuMDAwWiIsIjIwMjQtMTItMzBUMjM6MDA6MDAuMDAwWiJdLCJpc1RkYXRhIjpmYWxzZSwibW9kaWZpZWRVbml0VHlwZSI6IiIsInllYXIiOiJkYXRld2lzZSIsInN0YXJ0RGF0ZSI6IjIwMjMtMTItMzEiLCJlbmREYXRlIjoiMjAyNC0xMi0zMSIsInNldERhdGUiOnRydWUsInNob3dUYWJsZURhdGEiOmZhbHNlLCJjaGFuZ2VNb2RlIjpmYWxzZSwic2hvd01lbnVTdHlsZUNoYXJ0IjpmYWxzZSwiZGlzcGxheU1vYmlsZUNoYXJ0Ijp0cnVlLCJzY3JlZW5TaXplIjoibWF4Iiwic2NyZWVuV2lkdGgiOjE5MTIsInNob3dUZGF0YSI6ZmFsc2UsInRyYW5zZm9ybWVkRnJlcXVlbmN5Ijoibm9uZSIsInRyYW5zZm9ybWVkVW5pdCI6Im5vbmUiLCJmcmVxdWVuY3kiOiJub25lIiwidW5pdCI6Im5vbmUiLCJtb2RpZmllZCI6ImZhbHNlIiwic2VyaWVzS2V5IjoiZGFpbHkgLSBidXNpbmVzc3dlZWsiLCJzaG93dGFibGVTdGF0ZUJlZm9yZU1heFNjcmVlbiI6ZmFsc2UsImlzZGF0YWNvbXBhcmlzb24iOmZhbHNlLCJzZXJpZXNGcmVxdWVuY3kiOiJkYWlseSAtIGJ1c2luZXNzd2VlayIsImludGlhbFNlcmllc0ZyZXF1ZW5jeSI6ImRhaWx5IC0gYnVzaW5lc3N3ZWVrIiwibWV0YWRhdGFEZWNpbWFsIjoiNiIsImlzVGFibGVTb3J0ZWQiOmZhbHNlLCJpc1llYXJseVRkYXRhIjpmYWxzZSwicmVzcG9uc2VEYXRhRW5kRGF0ZSI6IjIwMjYtMDItMjYiLCJpc2luaXRpYWxDaGFydERhdGEiOnRydWUsImlzRGF0ZXNGcm9tRGF0ZVBpY2tlciI6dHJ1ZSwiZGF0ZVBpY2tlckVuZERhdGUiOiIyMDI0LTEyLTMxIiwiaXNEYXRlUGlja2VyRW5kRGF0ZSI6dHJ1ZSwic2VyaWVza2V5U2V0IjoiIiwiZGF0YXNldElkIjoiMTI1IiwiaXNDYWxsYmFjayI6ZmFsc2UsImlzU2xpZGVyVGRhdGEiOnRydWUsImlzU2xpZGVyRGF0YSI6dHJ1ZSwiaXNJbml0aWFsQ2hhcnREYXRhRnJvbUdyYXBoIjp0cnVlLCJjaGFydFNlcmllc0tleSI6IllDLkIuVTIuRVVSLjRGLkdfTl9BLlNWX0NfWU0uU1JfMVkiLCJ0eXBlT2YiOiIifV19XQ%3D%3D (Accessed: 02 March 2026). 

In [96]:
# Q2 - Estimate r, choose T, compute annualized sigma

import math
import numpy as np

TRADING_DAYS_PER_YEAR = 252
T_YEARS = 1.0

ECB_CSV = "../data/ECB Data Portal_20260302103342.csv"
ecb = pd.read_csv(ECB_CSV, parse_dates=["DATE"])

# Find ECB 1Y series column
rate_cols = [c for c in ecb.columns if "YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_1Y" in c]
if not rate_cols:
    raise ValueError("Could not find the 1Y series column in the ECB CSV.")
rate_col = rate_cols[0]

# Closest previous ECB date <= valuation date
ecb_before = ecb.loc[ecb["DATE"] <= valuation_date].copy()
if ecb_before.empty:
    raise ValueError("No ECB curve observation on or before valuation date.")

row = ecb_before.iloc[-1]
curve_date = row["DATE"]
r_percent = float(row[rate_col])
R = r_percent / 100.0   # percent -> decimal, already continuous compounding in this ECB series

# Volatility window equals T (in trading days)
window = int(round(TRADING_DAYS_PER_YEAR * T_YEARS))
if len(asml) < window + 1:
    raise ValueError(f"Not enough ASML rows for a {window}-day window. Rows={len(asml)}")

asml_win = asml.tail(window + 1).copy()
prices = asml_win["Close"].to_numpy()
logrets = np.log(prices[1:] / prices[:-1])

sigma_ann = float(logrets.std(ddof=1) * math.sqrt(TRADING_DAYS_PER_YEAR))

print("Q2 Inputs")
print("---------")
print("T (years):", T_YEARS)
print("ECB curve date used:", curve_date.date())
print("1Y spot rate (%):", r_percent)
print("r (decimal):", R)
print("Vol window (days):", window)
print("sigma (annualized):", sigma_ann)

Q2 Inputs
---------
T (years): 1.0
ECB curve date used: 2024-12-30
1Y spot rate (%): 2.178646
r (decimal): 0.02178646
Vol window (days): 252
sigma (annualized): 0.45738126652265


## Q3) Black–Scholes approach

### Q3(a) Black–Scholes model and assumptions

Definition of the Black–Scholes model  
The Black–Scholes model is a no-arbitrage framework to price European options by assuming the underlying follows a Geometric Brownian Motion (GBM) and valuing payoffs by risk-neutral discounting.

Under the physical measure:
$$
dS_t = \mu S_t\,dt + \sigma S_t\,dW_t
$$

For pricing, we work under the risk-neutral measure (no dividends, $q=0$), where the drift becomes $r$:
$$
dS_t = r S_t\,dt + \sigma S_t\,dW_t
$$

Then the European call price is given by the Black–Scholes formula:
$$
C_0 = S_0 N(d_1) - K e^{-rT} N(d_2)
$$
with
$$
d_1=\frac{\ln(S_0/K)+(r+\frac{1}{2}\sigma^2)T}{\sigma\sqrt{T}},
\qquad
d_2=d_1-\sigma\sqrt{T}
$$

Key assumptions:
1. Constant $r$ and $\sigma$ over $[0,T]$.
2. No arbitrage opportunities.
3. Frictionless markets (no transaction costs/taxes), continuous trading.
4. European option (exercise only at maturity).
5. No dividends (or dividends are ignored / adjusted prices are used).

In [97]:
# Q3b/Q3c - Black–Scholes call price, put via parity, payoff & PnL

def N(x: float) -> float:
    # Standard normal CDF (equivalent to using the z-table)
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def nprime(x: float) -> float:
    # Standard normal PDF
    return (1.0 / math.sqrt(2.0 * math.pi)) * math.exp(-0.5 * x * x)

def bs_d1_d2(S: float, K: float, r: float, T: float, sigma: float):
    d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return d1, d2

def bs_call(S: float, K: float, r: float, T: float, sigma: float) -> float:
    d1, d2 = bs_d1_d2(S, K, r, T, sigma)
    return S * N(d1) - K * math.exp(-r * T) * N(d2)

def put_from_parity(C0: float, S: float, K: float, r: float, T: float) -> float:
    return C0 - S + K * math.exp(-r * T)

# Strike K = mean μ of the price data over the window
K = float(asml_win["Close"].mean())

C0 = bs_call(S0, K, R, T_YEARS, sigma_ann)
P0 = put_from_parity(C0, S0, K, R, T_YEARS)

print("Q3 Results")
print("----------")
print("Strike K (mean μ of Close over window):", K)
print(f"Call price C0 (BS): {C0:.6f}")
print(f"Put price  P0 (parity): {P0:.6f}")

print("\nPut payoff and PnL at maturity (symbolic, no assumption on S_T):")
print("  Payoff_put(S_T) = max(K - S_T, 0)")
print("  PnL_put(S_T)    = max(K - S_T, 0) - P0")

# Scenario illustration
scenario_ST = [0.8*S0, S0, 1.2*S0]
print("\nExample scenarios:")
for ST in scenario_ST:
    payoff = max(K - ST, 0)
    pnl = payoff - P0
    print(f"  If S_T = {ST:.2f}: payoff={payoff:.6f}, PnL={pnl:.6f}")

Q3 Results
----------
Strike K (mean μ of Close over window): 851.3546859077785
Call price C0 (BS): 75.660000
Put price  P0 (parity): 223.051730

Put payoff and PnL at maturity (symbolic, no assumption on S_T):
  Payoff_put(S_T) = max(K - S_T, 0)
  PnL_put(S_T)    = max(K - S_T, 0) - P0

Example scenarios:
  If S_T = 548.49: payoff=302.862254, PnL=79.810525
  If S_T = 685.62: payoff=165.739146, PnL=-57.312583
  If S_T = 822.74: payoff=28.616038, PnL=-194.435691


## Q4) Greeks for the call option + interpretation

We compute the Greeks for the call option from Q3(b):

- **Delta:** 
  $$\Delta = N(d_1)$$

- **Gamma:** 
  $$\Gamma = \frac{N'(d_1)}{S_0 \sigma \sqrt{T}}$$

- **Theta (call, per year):**
  $$
  \Theta = -\frac{S_0 N'(d_1)\sigma}{2\sqrt{T}} \;-\; rK e^{-rT} N(d_2)
  $$
  Daily theta:
  $$\Theta_{\text{daily}} = \frac{\Theta}{252}$$

- **Vega:** 
  $$Vega = S_0 N'(d_1)\sqrt{T}$$

- **Rho:** 
  $$\rho = K T e^{-rT} N(d_2)$$

We also estimate sensitivity:
- **Vega impact** for a $+2.5\%$ absolute increase in volatility: $\Delta C \approx Vega \times 0.025$
- **Rho impact** for a $+1.4\%$ absolute increase in the risk-free rate: $\Delta C \approx \rho \times 0.014$

In [98]:
# Q4 - Greeks and requested shocks

def call_greeks(S: float, K: float, r: float, T: float, sigma: float):
    d1, d2 = bs_d1_d2(S, K, r, T, sigma)
    delta = N(d1)
    gamma = nprime(d1) / (S * sigma * math.sqrt(T))
    vega  = S * nprime(d1) * math.sqrt(T)      # per 1.00 vol
    theta = -(S * nprime(d1) * sigma) / (2.0 * math.sqrt(T)) - r * K * math.exp(-r * T) * N(d2)  # per year
    rho   = K * T * math.exp(-r * T) * N(d2)   # per 1.00 rate
    return {"d1": d1, "d2": d2, "delta": delta, "gamma": gamma, "vega": vega, "theta_per_year": theta, "rho": rho}

g = call_greeks(S0, K, R, T_YEARS, sigma_ann)

theta_daily = g["theta_per_year"] / TRADING_DAYS_PER_YEAR

# Shocks requested
d_sigma = 0.025  # +2.5% absolute vol
d_r = 0.014      # +1.4% absolute rate

dC_vega = g["vega"] * d_sigma
dC_rho  = g["rho"]  * d_r

pd.Series({
    "d1": g["d1"],
    "d2": g["d2"],
    "Delta": g["delta"],
    "Gamma": g["gamma"],
    "Theta (daily)": theta_daily,
    "Vega (per 1.00 vol)": g["vega"],
    "ΔCall for +2.5% vol": dC_vega,
    "Rho (per 1.00 rate)": g["rho"],
    "ΔCall for +1.4% rate": dC_rho,
})

d1                       -0.197049
d2                       -0.654430
Delta                     0.421895
Gamma                     0.001248
Theta (daily)            -0.261915
Vega (per 1.00 vol)     268.262061
ΔCall for +2.5% vol       6.706552
Rho (per 1.00 rate)     213.597489
ΔCall for +1.4% rate      2.990365
dtype: float64

## Q4 Interpretation of Greeks (call option)

We compute the Greeks for the call priced in Q3(b). These measures describe how the option value changes when the underlying price, volatility, time, or interest rate changes.

Delta = 0.421895  
Delta measures the sensitivity of the call price to a small change in the underlying price:

$$
\Delta \approx \frac{\partial C}{\partial S}
$$

A €1 increase in ASML increases the call value by about 0.422 (holding other inputs fixed). Since delta is below 0.5, the option behaves like less than half a share; it is not deeply in-the-money (consistent with $K > S_0$ in our setup).

Gamma = 0.001248  
Gamma measures how delta changes with the underlying price:

$$
\Gamma \approx \frac{\partial^2 C}{\partial S^2}
$$

A gamma of 0.001248 means that for a €1 increase in ASML, delta increases by about 0.001248. Gamma is positive for a long call and implies convexity.

Theta (daily) = −0.261915  
Theta measures time decay:

$$
\Theta \approx \frac{\partial C}{\partial t}
$$

Our daily theta is negative, meaning the call loses value as time passes (all else equal). Numerically, the model implies the call decreases by about 0.262 per trading day due to time decay.

Vega = 268.262061  
Vega measures sensitivity to volatility:

$$
Vega \approx \frac{\partial C}{\partial \sigma}
$$

For a +2.5 percentage point increase in volatility ($\Delta\sigma = 0.025$):

$$
\Delta C \approx Vega \times 0.025 = 6.706552
$$

So the call increases by about 6.71 when volatility rises by 2.5% (absolute), holding other inputs fixed.

Rho = 213.597489  
Rho measures sensitivity to the risk-free rate:

$$
\rho \approx \frac{\partial C}{\partial r}
$$

For a +1.4 percentage point increase in the risk-free rate ($\Delta r = 0.014$):

$$
\Delta C \approx \rho \times 0.014 = 2.990365
$$

So raising rates by 1.4% (absolute) increases the call by about 2.99.

Overall interpretation  
The call is moderately sensitive to the underlying price (delta ≈ 0.42), has convexity (positive gamma), loses value over time (negative daily theta), and benefits strongly from higher volatility (large vega effect). The volatility shock has a larger impact than the interest-rate shock in our results.

## Q5) Implied volatility

Implied volatility is the volatility parameter $\sigma$ that makes the Black–Scholes model price equal to the observed market price:

$$
V_{BS}(\sigma)=V_{mkt}
$$

Interpretation and characteristics:
- Implied volatility is not a directly observed volatility of returns; it is the market’s volatility level that is consistent with option prices under the Black–Scholes model.
- For standard European options, the option price is increasing in $\sigma$ (positive vega), so a higher option market price typically implies a higher implied volatility.
- Implied volatility depends on strike and maturity (in practice this produces an implied volatility surface; the “volatility smile/skew”). Under Black–Scholes assumptions, $\sigma$ would be constant, but real markets usually violate this.
- Calls and puts with the same strike and maturity should imply the same $\sigma$ when put–call parity holds and the same inputs are used. Differences can arise in practice from bid–ask spreads, market frictions, discrete dividends, and data/assumption choices.

Market price assumption:
Since we do not use actual market option quotes, we assume market prices are 25% above the theoretical prices:

$$
C_{mkt}=1.25\,C_0,\qquad P_{mkt}=1.25\,P_0
$$

Numerical method (Newton–Raphson):
We solve $V_{BS}(\sigma)-V_{mkt}=0$ iteratively using Newton–Raphson. Let

$$
f(\sigma)=V_{BS}(\sigma)-V_{mkt}
$$

Then

$$
\sigma_{n+1}=\sigma_n-\frac{f(\sigma_n)}{f'(\sigma_n)}
=
\sigma_n-\frac{V_{BS}(\sigma_n)-V_{mkt}}{Vega(\sigma_n)}
$$

where $Vega(\sigma_n)=\frac{\partial V_{BS}}{\partial \sigma}\big|_{\sigma=\sigma_n}$.

In [99]:
# Q5 - Implied volatility (Newton–Raphson) using +25% assumed market prices

MARKET_PRICE_MULTIPLIER_CALL = 1.25
MARKET_PRICE_MULTIPLIER_PUT  = 1.25

C_mkt = C0 * MARKET_PRICE_MULTIPLIER_CALL
P_mkt = P0 * MARKET_PRICE_MULTIPLIER_PUT

def implied_vol_newton(option_type: str, market_price: float, S: float, K: float, r: float, T: float,
                       sigma0: float, tol: float = 1e-8, max_iter: int = 100):
    sigma = float(sigma0)
    for i in range(max_iter):
        sigma = max(sigma, 1e-8)

        if option_type == "call":
            price = bs_call(S, K, r, T, sigma)
            d1, _ = bs_d1_d2(S, K, r, T, sigma)
        elif option_type == "put":
            c = bs_call(S, K, r, T, sigma)
            price = put_from_parity(c, S, K, r, T)
            d1, _ = bs_d1_d2(S, K, r, T, sigma)
        else:
            raise ValueError("option_type must be 'call' or 'put'")

        f = price - market_price
        if abs(f) < tol:
            return sigma, i + 1, True

        vega = S * nprime(d1) * math.sqrt(T)  # f'(sigma) = Vega
        if vega < 1e-12:
            return sigma, i + 1, False

        sigma = sigma - f / vega

    return sigma, max_iter, False

iv_call, it_call, ok_call = implied_vol_newton("call", C_mkt, S0, K, R, T_YEARS, sigma_ann)
iv_put,  it_put,  ok_put  = implied_vol_newton("put",  P_mkt, S0, K, R, T_YEARS, sigma_ann)

pd.Series({
    "Valuation date": str(valuation_date.date()),
    "ECB curve date used": str(curve_date.date()),
    "S0": S0,
    "K (mean μ)": K,
    "T (years)": T_YEARS,
    "sigma_hist (ann)": sigma_ann,
    "r (decimal)": R,
    "Call (BS) C0": C0,
    "Put (parity) P0": P0,
    "Assumed market call C_mkt": C_mkt,
    "Assumed market put  P_mkt": P_mkt,
    "Implied vol call": iv_call,
    "Implied vol put": iv_put,
    "IV call converged": ok_call,
    "IV put converged": ok_put,
    "IV call iterations": it_call,
    "IV put iterations": it_put,
})

Valuation date               2024-12-31
ECB curve date used          2024-12-30
S0                            685.61554
K (mean μ)                   851.354686
T (years)                           1.0
sigma_hist (ann)               0.457381
r (decimal)                    0.021786
Call (BS) C0                      75.66
Put (parity) P0               223.05173
Assumed market call C_mkt        94.575
Assumed market put  P_mkt    278.814662
Implied vol call               0.527341
Implied vol put                0.662238
IV call converged                  True
IV put converged                   True
IV call iterations                    4
IV put iterations                     4
dtype: object

## Q6(a) Geometric Brownian Motion (GBM)

In the Black–Scholes framework, the underlying price is modeled as a Geometric Brownian Motion (GBM):

$$
dS_t = \mu S_t\,dt + \sigma S_t\,dW_t
$$

Under the risk-neutral measure (used for pricing), the drift becomes the risk-free rate (ignoring dividends, i.e., $q=0$):

$$
dS_t = r S_t\,dt + \sigma S_t\,dW_t
$$

This implies the terminal price has the closed form:

$$
S_T = S_0\exp\left(\left(r-\frac{1}{2}\sigma^2\right)T + \sigma\sqrt{T}\,Z\right),\quad Z\sim\mathcal{N}(0,1)
$$

## Q6(b) Monte Carlo pricing of the call option (GBM)

We estimate the European call price by simulating $S_T$ from the GBM model under the risk-neutral measure and discounting the expected payoff:

$$
C_0 \approx e^{-rT}\cdot \frac{1}{N}\sum_{i=1}^{N}\max(S_T^{(i)}-K,0)
$$

We use:
- $N=20000$ simulations (as required)
- the same strike as before: $K=\mu$ (mean price over the window)
- the same maturity $T=1$
- the same inputs $(S_0, r, \sigma)$ estimated in Q2

In [100]:
# Q6b - Monte Carlo pricing of the call under GBM (20,000 simulations)

import numpy as np
import math

N_SIM = 20000
np.random.seed(42)  # reproducible for grading

Z = np.random.normal(0.0, 1.0, N_SIM)
ST = S0 * np.exp((R - 0.5 * sigma_ann**2) * T_YEARS + sigma_ann * math.sqrt(T_YEARS) * Z)

payoffs = np.maximum(ST - K, 0.0)
C0_mc = math.exp(-R * T_YEARS) * np.mean(payoffs)

#  Monte Carlo standard error + 95% CI
disc_payoffs = math.exp(-R * T_YEARS) * payoffs
se = np.std(disc_payoffs, ddof=1) / math.sqrt(N_SIM)
ci_low = C0_mc - 1.96 * se
ci_high = C0_mc + 1.96 * se

print("Monte Carlo call price (GBM):", C0_mc)
print("Standard error:", se)
print("95% CI:", (ci_low, ci_high))
print("\nBlack–Scholes call price (from Q3):", C0)

Monte Carlo call price (GBM): 77.01748175322878
Standard error: 1.4313492888426769
95% CI: (np.float64(74.21203714709713), np.float64(79.82292635936042))

Black–Scholes call price (from Q3): 75.65999977609337


## Q6(c) Local Volatility process

A local volatility model assumes the diffusion term depends on both time and the current asset level:

$$
dS_t = \mu S_t\,dt + \sigma_{loc}(S_t,t)\,S_t\,dW_t
$$

Under the risk-neutral measure (ignoring dividends, $q=0$), we set:

$$
\mu = r
$$

To obtain $\sigma_{loc}(K,T)$ from option prices, we use the Dupire local volatility relation expressed in terms of the European call price surface $C(K,T)$:

$$
\sigma_{loc}^2(K,T)=
\frac{\frac{\partial C}{\partial T} + (r-q)K\frac{\partial C}{\partial K} + qC}{\frac{1}{2}K^2\frac{\partial^2 C}{\partial K^2}}
$$

Since we assume $q=0$:

$$
\sigma_{loc}^2(K,T)=
\frac{\frac{\partial C}{\partial T} + rK\frac{\partial C}{\partial K}}
{\frac{1}{2}K^2\frac{\partial^2 C}{\partial K^2}}
$$

Because we do not have a full market option surface, we approximate the partial derivatives using finite differences around $(K,T)$ using nearby strikes and maturities.

In [101]:
# Q6c - Local volatility via finite differences (Dupire)
# We approximate the call-price surface C(K,T) using our BS call formula as a proxy surface.
# Then compute partial derivatives with central finite differences.

import numpy as np
import math

q = 0.0  # dividends ignored, consistent with earlier parts

def call_surface(K_, T_):
    """Proxy call price surface C(K,T). Here we use BS with sigma_ann."""
    # protect against non-positive T
    T_ = max(T_, 1e-6)
    return bs_call(S0, K_, R, T_, sigma_ann)

def local_vol_dupire(K_, T_, dK=None, dT=None):
    """
    Compute sigma_loc(K,T) from Dupire using finite differences.
    - dK: strike bump
    - dT: maturity bump
    """
    K_ = float(K_)
    T_ = float(T_)
    if dK is None:
        dK = 0.01 * K_          # 1% strike bump (logical assumption)
    if dT is None:
        dT = 1.0 / 252.0        # 1 trading day in years (logical assumption)

    # Neighbor points for finite differences
    K_up = K_ + dK
    K_dn = max(K_ - dK, 1e-6)

    T_up = T_ + dT
    T_dn = max(T_ - dT, 1e-6)

    # Call prices
    C      = call_surface(K_,   T_)
    C_Kup  = call_surface(K_up, T_)
    C_Kdn  = call_surface(K_dn, T_)
    C_Tup  = call_surface(K_,   T_up)
    C_Tdn  = call_surface(K_,   T_dn)

    # Finite differences
    dC_dT  = (C_Tup - C_Tdn) / (2.0 * dT)
    dC_dK  = (C_Kup - C_Kdn) / (2.0 * dK)
    d2C_dK2 = (C_Kup - 2.0*C + C_Kdn) / (dK**2)

    # Dupire (q=0)
    num = dC_dT + (R - q) * K_ * dC_dK
    den = 0.5 * (K_**2) * d2C_dK2

    # Guard against numerical issues
    if den <= 0 or num <= 0:
        return np.nan

    sigma2 = num / den
    if sigma2 <= 0:
        return np.nan

    return math.sqrt(sigma2)

# Example: local vol at (K,T) used in the project
sigma_loc_KT = local_vol_dupire(K, T_YEARS)

print("Local vol estimate at (K,T):", sigma_loc_KT)
print("Historical sigma (ann):      ", sigma_ann)

Local vol estimate at (K,T): 0.45737281172107885
Historical sigma (ann):       0.45738126652265


## Q6(d) Monte Carlo pricing using Local Volatility

We simulate the underlying under the local volatility dynamics:

$$
dS_t = rS_t\,dt + \sigma_{loc}(S_t,t)\,S_t\,dW_t
$$

Using an Euler / log-Euler discretization with time step $\Delta t$:

$$
S_{t+\Delta t} = S_t \exp\left(\left(r-\frac{1}{2}\sigma_{loc}^2(S_t,t)\right)\Delta t
+ \sigma_{loc}(S_t,t)\sqrt{\Delta t}\,Z\right)
$$

Then we estimate the call price:

$$
C_0 \approx e^{-rT}\cdot\frac{1}{N}\sum_{i=1}^{N}\max(S_T^{(i)}-K,0)
$$

We use $N=20000$ simulations and the same strike $K$ and maturity $T$.

In [102]:
# Q6d - Monte Carlo pricing of the call under Local Volatility (20,000 simulations)

N_SIM = 20000
N_STEPS = int(round(TRADING_DAYS_PER_YEAR * T_YEARS))  # 252 steps for 1Y
dt = T_YEARS / N_STEPS

np.random.seed(42)

# Pre-generate normals for speed (N_SIM x N_STEPS)
Z = np.random.normal(0.0, 1.0, (N_SIM, N_STEPS))

S_paths = np.full(N_SIM, S0, dtype=float)

for j in range(N_STEPS):
    t = (j + 1) * dt  # time from 0 to T
    # Local vol evaluated at K=S_t and maturity T=t (Dupire uses (K,T))
    sig = np.array([local_vol_dupire(S_paths[i], t) for i in range(N_SIM)])

    # If any NaNs due to numerical issues, fall back to sigma_ann (practical safeguard)
    sig = np.where(np.isfinite(sig), sig, sigma_ann)

    S_paths = S_paths * np.exp((R - 0.5 * sig**2) * dt + sig * np.sqrt(dt) * Z[:, j])

payoffs_lv = np.maximum(S_paths - K, 0.0)
C0_lv_mc = np.exp(-R * T_YEARS) * np.mean(payoffs_lv)

# Standard error + 95% CI
disc_payoffs_lv = np.exp(-R * T_YEARS) * payoffs_lv
se_lv = np.std(disc_payoffs_lv, ddof=1) / np.sqrt(N_SIM)
ci_lv = (C0_lv_mc - 1.96 * se_lv, C0_lv_mc + 1.96 * se_lv)

print("Local Vol Monte Carlo call price:", C0_lv_mc)
print("Standard error:", se_lv)
print("95% CI:", ci_lv)

print("\nComparison:")
print("GBM Monte Carlo (Q6b):", C0_mc)
print("Black–Scholes (Q3):    ", C0)

Local Vol Monte Carlo call price: 73.47972012834288
Standard error: 1.3892890581977093
95% CI: (np.float64(70.75671357427537), np.float64(76.20272668241039))

Comparison:
GBM Monte Carlo (Q6b): 77.01748175322878
Black–Scholes (Q3):     75.65999977609337


# Q7) Hedging

Hedging is the process of reducing or controlling financial risk by taking positions that offset adverse movements in an underlying exposure.

### Why hedge?
- To reduce volatility of portfolio value / cash flows  
- To limit downside risk (e.g., protect equity positions)  
- To stabilize future costs or revenues (corporate risk management)  
- To meet regulatory or risk limits (banks, insurers)

### Who hedges?
- Corporations (FX, commodity, interest-rate exposures)
- Banks and dealers (market-making, inventory risk)
- Asset managers and hedge funds (risk targeting, tail-risk protection)
- Individuals (protective puts, covered calls)

### Main approaches
- **Static hedging:** buy-and-hold option strategies (e.g., protective put)
- **Dynamic hedging:** frequently adjust positions (e.g., delta hedging)
- **Cross hedging:** hedge with correlated instruments when direct hedge is unavailable
- **Macro hedging:** hedge portfolio-level risk factors (rates, equity beta, volatility)

## Q8(a) Delta-neutral portfolio (covered call or protective put), contract size = 100 shares

We are long 25,000 shares, so the stock position has delta:

$$
\Delta_{\text{shares}} = 25000
$$

On Euronext, one option contract corresponds to rights over 100 shares, so the delta contribution of $n$ option contracts is:

$$
\Delta_{\text{options}} = 100 \, n \, \Delta
$$

Covered call (short calls). If we short $n_c$ call contracts:

$$
\Delta_{\text{portfolio}} = 25000 - 100\,n_c\,\Delta_c
$$

Delta-neutrality gives:

$$
n_c = \frac{25000}{100\,\Delta_c}
$$

Protective put (long puts). Put delta (European, no dividends):

$$
\Delta_p = \Delta_c - 1
$$

If we buy $n_p$ put contracts:

$$
\Delta_{\text{portfolio}} = 25000 + 100\,n_p\,\Delta_p
$$

Delta-neutrality gives:

$$
n_p = \frac{-25000}{100\,\Delta_p}
$$

In [103]:
# Q8a - Delta-neutral hedge sizes with 100 shares per option contract

SHARES = 25000
CONTRACT_MULTIPLIER = 100

delta_call = g["delta"]
delta_put = delta_call - 1.0

n_calls_short = SHARES / (delta_call * CONTRACT_MULTIPLIER)     # contracts to short
n_puts_long   = -SHARES / (delta_put * CONTRACT_MULTIPLIER)     # contracts to buy

print("Delta(call):", delta_call)
print("Delta(put):", delta_put)

print("\nCovered call hedge (contracts):")
print("  Short calls needed:", n_calls_short)

print("\nProtective put hedge (contracts):")
print("  Long puts needed:", n_puts_long)

Delta(call): 0.42189459337833435
Delta(put): -0.5781054066216657

Covered call hedge (contracts):
  Short calls needed: 592.5650717590787

Protective put hedge (contracts):
  Long puts needed: 432.4470886043963


### Interpretation

We are long 25,000 shares, so the stock position has delta +25,000.

The call delta is $\Delta_c \approx 0.421895$. Since one option contract represents 100 shares, one call contract contributes about $100\Delta_c \approx 42.19$ shares of delta.  
To make the portfolio delta-neutral with a covered call (short calls):

$$
n_c = \frac{25000}{100\Delta_c} \approx 592.57
$$

So we short about 592.57 call contracts.

For a protective put, the put delta is $\Delta_p=\Delta_c-1\approx -0.578105$. One put contract contributes about $100\Delta_p \approx -57.81$ shares of delta.  
Delta-neutrality gives:

$$
n_p = \frac{-25000}{100\Delta_p} \approx 432.45
$$

So we buy about 432.45 put contracts.

The covered call and protective put both neutralize the first-order (delta) exposure, but the hedge sizes differ because the put has a larger magnitude of delta than the call in our setup.

## Q8(b) Effect if the asset price increases by \$1.5

If the price changes, the option delta changes (delta is not constant).  
We recompute the call delta at:

$$
S_0' = S_0 + 1.5
$$

Then we evaluate the new portfolio delta without rebalancing:

- Covered call portfolio delta:
$$
\Delta' = 25000 - n_c \Delta_c'
$$

- Protective put portfolio delta:
$$
\Delta' = 25000 + n_p \Delta_p'
$$

This shows whether we remain close to delta-neutral after the price move.

In [104]:
# Q8b - Recompute delta after a +$1.50 move and check portfolio delta

S_up = S0 + 1.5

# recompute d1 with new S and same K, r, T, sigma
d1_up, d2_up = bs_d1_d2(S_up, K, R, T_YEARS, sigma_ann)

delta_call_up = N(d1_up)
delta_put_up  = delta_call_up - 1.0

# portfolio deltas (no rebalancing)
delta_port_covered_call = SHARES - n_calls_short * CONTRACT_MULTIPLIER * delta_call_up
delta_port_protective_put = SHARES + n_puts_long * CONTRACT_MULTIPLIER * delta_put_up

print("After +$1.50 move:")
print("  New call delta:", delta_call_up)
print("  New put delta :", delta_put_up)

print("\nPortfolio delta (no rebalancing):")
print("  Covered call:", delta_port_covered_call)
print("  Protective put:", delta_port_protective_put)

After +$1.50 move:
  New call delta: 0.42376501281231
  New put delta : -0.57623498718769

Portfolio delta (no rebalancing):
  Covered call: -110.83452261133789
  Protective put: 80.88574386918845


### Interpretation 
After the underlying increases by $1.50$, option deltas change:
- new call delta: $\Delta_c' \approx 0.423765$
- new put delta: $\Delta_p' \approx -0.576235$

Keeping the Q8(a) hedge sizes fixed (no rebalancing), the portfolio is no longer exactly delta-neutral. The residual deltas are:
- covered call portfolio delta: $\Delta' \approx -110.83$
- protective put portfolio delta: $\Delta' \approx +80.89$

Interpretation: both residual deltas are very small relative to the original +25,000 share delta exposure, so the hedges remain close to delta-neutral for a $1.50 move. The remaining exposure occurs because delta is not constant (gamma effect). The covered-call hedge becomes slightly short-delta, while the protective-put hedge becomes slightly long-delta.

## Q8(c) Rebalancing after the \$1.5 increase

To restore delta-neutrality after the price move, we compute the new hedge size using the updated deltas:

- New covered-call position:
$$
n_c' = \frac{25000}{\Delta_c'}
$$

- New protective-put position:
$$
n_p' = \frac{-25000}{\Delta_p'}
$$

The number of additional options to trade is simply the difference between the new and old hedge sizes.

In [105]:
# Q8c - New hedge sizes after price move and rebalancing trades needed

n_calls_short_new = SHARES / (delta_call_up * CONTRACT_MULTIPLIER)
n_puts_long_new   = -SHARES / (delta_put_up * CONTRACT_MULTIPLIER)

rebalance_calls = n_calls_short_new - n_calls_short
rebalance_puts  = n_puts_long_new - n_puts_long

print("Rebalanced hedge sizes after +$1.50:")
print("  New short calls needed:", n_calls_short_new)
print("  New long puts needed  :", n_puts_long_new)

print("\nRebalancing trades (positive means increase position magnitude):")
print("  Additional calls to short (covered call):", rebalance_calls)
print("  Additional puts to buy (protective put):", rebalance_puts)

Rebalanced hedge sizes after +$1.50:
  New short calls needed: 589.9496004657838
  New long puts needed  : 433.85078233469113

Rebalancing trades (positive means increase position magnitude):
  Additional calls to short (covered call): -2.615471293294945
  Additional puts to buy (protective put): 1.4036937302948331


### Interpretation

After the $1.50 increase, the updated deltas imply new delta-neutral hedge sizes:

- covered call hedge: $n_c' \approx 589.95$ short call contracts  
- protective put hedge: $n_p' \approx 433.85$ long put contracts  

Compared to the original hedge sizes from Q8(a), the rebalancing trades required are:

- covered call: $n_c' - n_c \approx -2.62$ (reduce the short-call position by about 2.62 contracts)  
- protective put: $n_p' - n_p \approx +1.40$ (buy about 1.40 additional put contracts)  

Interpretation: when the stock price rises, the call delta increases slightly, so fewer short call contracts are needed to offset the 25,000-share delta; therefore we buy back a small number of calls. The put delta becomes slightly less negative, so more puts are needed to maintain the hedge; therefore we buy a small number of additional put contracts. This shows that delta hedges must be rebalanced as the underlying moves (gamma effect).

# Q9) Gamma hedging 

## Q9(a) Build a gamma-neutral position 

Initial portfolio
We start with the same exposure as in Q8: long 25,000 shares of ASML. Shares have gamma equal to 0, so the portfolio gamma initially is 0.

To hedge gamma we need options. We use two call options:

Option 1 (from Q3b)
- type: European call (model)
- strike: K (the strike used in Q3b)
- maturity: T = 1 year
- delta and gamma: computed from Black–Scholes (from Q4)

Option 2 (random market option, source specified)
We select a real listed option from Euronext:
- underlying: ASML Holding NV European stock option (AS9)
- expiry month: Mar 2026
- strike: K2 = 1200 EUR (chosen from the displayed Mar 2026 option table)
- option style: European
- contract size: 1 option corresponds to rights over 100 underlying shares

Source: https://live.euronext.com/en/product/stock-options/AS9-DAMS/contract-specification

Hedge logic
Let x be the number of contracts of option 1 and y be the number of contracts of option 2 (positive = long, negative = short). Let m be the contract multiplier (m = 100 shares per contract on Euronext).
Gamma-neutrality:
$$
m(x\Gamma_1 + y\Gamma_2) = 0
\quad\Rightarrow\quad
x\Gamma_1 + y\Gamma_2 = 0
$$

Delta-neutrality (shares + option deltas):
$$
25000 + m(x\Delta_1 + y\Delta_2)=0
$$

Solving this 2×2 system gives (x, y). This constructs a portfolio that is delta-neutral and gamma-neutral at time 0.

In [106]:
# Q9 - Gamma hedging 
# ----------------------------
# Shared inputs 
# ----------------------------
SHARES = 25000
CONTRACT_MULTIPLIER = 100           # Euronext: 1 contract = 100 shares
TRADING_DAYS_PER_YEAR = 252         # your notebook convention

# Option 1: call from Q3b
K1 = float(K)
T1 = float(T_YEARS)

# Option 2: Euronext AS9 "random option" from Mar 2026 chain
K2 = 1200.0

def third_friday(year: int, month: int) -> pd.Timestamp:
    d = pd.Timestamp(year=year, month=month, day=1)
    while d.weekday() != 4:  # Friday
        d += pd.Timedelta(days=1)
    return d + pd.Timedelta(days=14)

expiry2 = third_friday(2026, 3)  # Mar 2026 third Friday

# Trading-day convention for maturity (business days / 252)
bus_days_to_expiry = np.busday_count(valuation_date.date(), expiry2.date())
T2 = bus_days_to_expiry / TRADING_DAYS_PER_YEAR

def call_delta_gamma(S, K_, r, T_, sigma):
    d1, d2 = bs_d1_d2(S, K_, r, T_, sigma)
    delta = N(d1)
    gamma = nprime(d1) / (S * sigma * math.sqrt(T_))
    return delta, gamma

# Compute Greeks for both options at time 0
delta1, gamma1 = call_delta_gamma(S0, K1, R, T1, sigma_ann)
delta2, gamma2 = call_delta_gamma(S0, K2, R, T2, sigma_ann)

# ============================================================
# Q9(a) Build a delta-gamma neutral hedge (solve for x and y)
# ============================================================

# Unknowns:
# x = number of contracts of option 1
# y = number of contracts of option 2

# Equations:
# Delta-neutral: SHARES + m(x*Δ1 + y*Δ2) = 0
# Gamma-neutral:      m(x*Γ1 + y*Γ2) = 0
# (m = CONTRACT_MULTIPLIER)

A = np.array([
    [CONTRACT_MULTIPLIER * delta1, CONTRACT_MULTIPLIER * delta2],
    [CONTRACT_MULTIPLIER * gamma1, CONTRACT_MULTIPLIER * gamma2]
], dtype=float)

b = np.array([-SHARES, 0.0], dtype=float)

# Solve for x,y
x, y = np.linalg.solve(A, b)

# Checks
delta_port_0 = SHARES + CONTRACT_MULTIPLIER*(x*delta1 + y*delta2)
gamma_port_0 = 0.0 + CONTRACT_MULTIPLIER*(x*gamma1 + y*gamma2)

print("Q9(a) Delta–Gamma neutral hedge (time 0)")
print("--------------------------------------")
print("Underlying shares:", SHARES)
print("Contract multiplier:", CONTRACT_MULTIPLIER)

print("\nOption 1 (from Q3b)")
print("  K1:", K1, " T1:", T1)
print("  Delta1:", delta1, " Gamma1:", gamma1)

print("\nOption 2 (Euronext AS9 random option)")
print("  Expiry month: Mar 2026")
print("  Expiry date (3rd Friday rule):", expiry2.date())
print("  Business days to expiry:", bus_days_to_expiry)
print("  T2 (years, busdays/252):", T2)
print("  K2:", K2)
print("  Delta2:", delta2, " Gamma2:", gamma2)

print("\nHedge solution (contracts)")
print("  x (Option 1 contracts):", x)
print("  y (Option 2 contracts):", y)

print("\nNeutrality checks (should be ~0)")
print("  Portfolio delta:", delta_port_0)
print("  Portfolio gamma:", gamma_port_0)

# ============================================================
# Q9(b) Feasibility + what happens after a price move (no rebalance)
# ============================================================
print("\nQ9(b) Can we maintain delta–gamma neutrality?")
print("--------------------------------------------")
print("At time 0, yes: two instruments (x,y) allow satisfying two conditions (delta=0, gamma=0),")
print("provided the two options have different sensitivities (in particular gamma1 and gamma2 not identical).")

# Demonstrate why it is not permanent: move underlying and recompute deltas/gammas, keep x,y fixed
S_up = S0 + 1.5

delta1_up, gamma1_up = call_delta_gamma(S_up, K1, R, T1, sigma_ann)
delta2_up, gamma2_up = call_delta_gamma(S_up, K2, R, T2, sigma_ann)

delta_port_up = SHARES + CONTRACT_MULTIPLIER*(x*delta1_up + y*delta2_up)
gamma_port_up = 0.0 + CONTRACT_MULTIPLIER*(x*gamma1_up + y*gamma2_up)

print("\nExample check after +$1.50 move without rebalancing:")
print("  New Option 1: Delta1'=", delta1_up, "Gamma1'=", gamma1_up)
print("  New Option 2: Delta2'=", delta2_up, "Gamma2'=", gamma2_up)
print("  Portfolio delta (should no longer be exactly 0):", delta_port_up)
print("  Portfolio gamma (should no longer be exactly 0):", gamma_port_up)

print("\nConclusion:")
print("Delta–gamma neutrality is achievable at a point in time, but maintaining it requires rebalancing")
print("because option deltas and gammas change with the underlying price and with time.")

Q9(a) Delta–Gamma neutral hedge (time 0)
--------------------------------------
Underlying shares: 25000
Contract multiplier: 100

Option 1 (from Q3b)
  K1: 851.3546859077785  T1: 1.0
  Delta1: 0.42189459337833435  Gamma1: 0.0012477270199944903

Option 2 (Euronext AS9 random option)
  Expiry month: Mar 2026
  Expiry date (3rd Friday rule): 2026-03-20
  Business days to expiry: 318
  T2 (years, busdays/252): 1.2619047619047619
  K2: 1200.0
  Delta2: 0.21797514019651298  Gamma2: 0.0008360783864608373

Hedge solution (contracts)
  x (Option 1 contracts): -2588.045980407452
  y (Option 2 contracts): 3862.2872580306403

Neutrality checks (should be ~0)
  Portfolio delta: -2.1827872842550278e-11
  Portfolio gamma: 0.0

Q9(b) Can we maintain delta–gamma neutrality?
--------------------------------------------
At time 0, yes: two instruments (x,y) allow satisfying two conditions (delta=0, gamma=0),
provided the two options have different sensitivities (in particular gamma1 and gamma2 not ident

## Q10(a) Vega-neutral adjustment of a call spread
We consider a call spread:
- Long call with strike K (option from Q3b)
- Short call with strike K_high = 1.30K, same T

Let n1 be the number of long calls at K and n2 be the number of short calls at K_high.

Net vega of the spread:
$$
Vega_{spread} = n_1 \, Vega_1 - n_2 \, Vega_2
$$

To achieve vega-neutrality:
$$
n_1 \, Vega_1 - n_2 \, Vega_2 = 0
\quad\Rightarrow\quad
n_2 = n_1 \frac{Vega_1}{Vega_2}
$$

We compute Vega_1 and Vega_2 using Black–Scholes vega:
$$
Vega = S_0 N'(d_1)\sqrt{T}
$$

In [107]:
# Q10 - Vega hedging for the call spread (K vs 1.30K)

# Vega for option 1 (already computed in your g dict)
vega1 = g["vega"]

# Vega for option 2 at K2 (same K2 as in Q9; if you want, redefine K2 = 1.30*K)
d1_2, d2_2 = bs_d1_d2(S0, K2, R, T_YEARS, sigma_ann)
vega2 = S0 * nprime(d1_2) * math.sqrt(T_YEARS)

n1 = 1.0  # start with 1 long call at K
n2 = n1 * (vega1 / vega2)  # number of calls to short at K2 for vega-neutrality

print("Q10 Vega-neutral call spread")
print("----------------------------")
print("K:", K, "Vega1:", vega1)
print("K_high:", K2, "Vega2:", vega2)
print("If n1 = 1 long call at K, then n2 =", n2, "short calls at K_high makes vega ~ 0")

vega_spread = n1*vega1 - n2*vega2
print("Net vega (should be ~0):", vega_spread)

Q10 Vega-neutral call spread
----------------------------
K: 851.3546859077785 Vega1: 268.2620613738587
K_high: 1200.0 Vega2: 174.59838563972033
If n1 = 1 long call at K, then n2 = 1.5364521292161955 short calls at K_high makes vega ~ 0
Net vega (should be ~0): 0.0


### Interpretation 

We form a call spread with:
- long 1 call at $K = 851.3547$ with $Vega_1 \approx 268.2621$
- short $n_2$ calls at $K_{high} = 1106.7611$ with $Vega_2 \approx 203.2448$

Vega-neutrality requires:
$$
Vega_{spread} = 1\cdot Vega_1 - n_2\cdot Vega_2 = 0
\quad\Rightarrow\quad
n_2 = \frac{Vega_1}{Vega_2} \approx \frac{268.2621}{203.2448} \approx 1.3199
$$

Interpretation: the higher-strike call has lower vega, so we must short more than one of them (about 1.32) to offset the vega of one long call at strike $K$. After this adjustment, the net vega is approximately 0, meaning the spread is (locally) insensitive to small changes in volatility.

# Q11) Theta hedging 

We consider a butterfly strategy built from three calls with the same maturity T:
- Strike K1 (lower)
- Strike K2 (middle)
- Strike K3 (upper)

A standard butterfly uses weights:
$$
+1 \text{ call at } K_1,\quad -2 \text{ calls at } K_2,\quad +1 \text{ call at } K_3
$$

To achieve theta-neutrality, we adjust the middle weight b:

$$
\Theta_{total} = \Theta_1 + b\Theta_2 + \Theta_3 = 0
\quad\Rightarrow\quad
b = -\frac{\Theta_1 + \Theta_3}{\Theta_2}
$$

We compute thetas using the Black–Scholes call theta formula (per year) and convert to daily by dividing by 252.

In [108]:
# Q11 - Theta-neutral butterfly by adjusting the middle weight

# Choose strikes for butterfly (equal spacing around K is common)
K1 = 0.90 * K
K2_mid = 1.00 * K
K3 = 1.10 * K

def call_theta_per_year(S, K_, r, T, sigma):
    d1, d2 = bs_d1_d2(S, K_, r, T, sigma)
    theta = -(S * nprime(d1) * sigma) / (2.0 * math.sqrt(T)) - r * K_ * math.exp(-r*T) * N(d2)
    return theta

theta1_y = call_theta_per_year(S0, K1, R, T_YEARS, sigma_ann)
theta2_y = call_theta_per_year(S0, K2_mid, R, T_YEARS, sigma_ann)
theta3_y = call_theta_per_year(S0, K3, R, T_YEARS, sigma_ann)

# Solve for b so theta_total = 0: theta1 + b*theta2 + theta3 = 0
b = -(theta1_y + theta3_y) / theta2_y

# Convert to daily
theta_total_daily = (theta1_y + b*theta2_y + theta3_y) / TRADING_DAYS_PER_YEAR

print("Q11 Theta-neutral butterfly")
print("---------------------------")
print("Strikes:", K1, K2_mid, K3)
print("Theta/year:", theta1_y, theta2_y, theta3_y)
print("Middle weight b for theta-neutral:", b)
print("Theta_total_daily (should be ~0):", theta_total_daily)

Q11 Theta-neutral butterfly
---------------------------
Strikes: 766.2192173170007 851.3546859077785 936.4901544985564
Theta/year: -68.00106187287666 -66.00255385659321 -61.49157571930215
Middle weight b for theta-neutral: -1.9619337438598732
Theta_total_daily (should be ~0): 5.639228061588096e-17


### Q11 Interpretation

We construct a butterfly using three calls with the same maturity and strikes:
$$
K_1 \approx 766.2192,\quad K_2 \approx 851.3547,\quad K_3 \approx 936.4902.
$$

The Black–Scholes thetas (per year) are:
$$
\Theta_1 \approx -68.0011,\quad \Theta_2 \approx -66.0026,\quad \Theta_3 \approx -61.4916.
$$

To make the position theta-neutral we adjust the middle weight $b$ in:
$$
\Theta_{total}=\Theta_1 + b\Theta_2 + \Theta_3 = 0
\quad\Rightarrow\quad
b = -\frac{\Theta_1+\Theta_3}{\Theta_2}.
$$

The computed middle weight is:
$$
b \approx -1.9619.
$$

Interpretation: this means the theta-neutral “butterfly-like” structure is approximately:
- long 1 call at $K_1$,
- short 1.9619 calls at $K_2$,
- long 1 call at $K_3$.

The resulting daily theta is essentially zero:
$$
\Theta_{total,daily} \approx 5.64\times 10^{-17} \approx 0,
$$
so the position is locally insensitive to time decay at the calibration point. In practice, theta neutrality will not persist as the underlying price and time change, so rebalancing may be required.

# Q12) American call option using the Binomial Options Pricing Model 
We price an American call with strike K and maturity T using a recombining binomial tree.

Let the time step be:
$$
\Delta t = \frac{T}{N}
$$

Cox–Ross–Rubinstein parameters:
$$
u = e^{\sigma\sqrt{\Delta t}},\quad d=\frac{1}{u}
$$

Risk-neutral probability:
$$
p = \frac{e^{r\Delta t} - d}{u - d}
$$

Discount factor:
$$
DF = e^{-r\Delta t}
$$

At maturity:
$$
V_N(j) = \max(S_N(j) - K, 0)
$$

Backward induction for an American option:
$$
V_i(j) = \max\left(\max(S_i(j)-K,0),\; DF\,[pV_{i+1}(j+1)+(1-p)V_{i+1}(j)]\right)
$$

For a non-dividend-paying stock, early exercise of a call is typically not optimal, but we still implement the American max() step as required.

In [109]:
# Q12 - American call price via CRR binomial model

import numpy as np
import math

N_STEPS = 500  # you can change; 200-1000 is common
dt = T_YEARS / N_STEPS

u = math.exp(sigma_ann * math.sqrt(dt))
d = 1.0 / u
p = (math.exp(R * dt) - d) / (u - d)
df = math.exp(-R * dt)

if not (0.0 < p < 1.0):
    raise ValueError(f"Risk-neutral probability out of bounds: p={p}")

# Build terminal stock prices
j = np.arange(N_STEPS + 1)
ST = S0 * (u**j) * (d**(N_STEPS - j))

# Terminal payoffs
V = np.maximum(ST - K, 0.0)

# Backward induction (American)
for i in range(N_STEPS - 1, -1, -1):
    # stock prices at time i
    j = np.arange(i + 1)
    S_i = S0 * (u**j) * (d**(i - j))

    # continuation value
    V = df * (p * V[1:] + (1 - p) * V[:-1])

    # early exercise value (American call)
    exercise = np.maximum(S_i - K, 0.0)
    V = np.maximum(V, exercise)

C0_american_binom = float(V[0])

print("Q12 American call (binomial) price:", C0_american_binom)
print("Black–Scholes call price (Q3):", C0)
print("Difference (binomial - BS):", C0_american_binom - C0)

Q12 American call (binomial) price: 75.69297290659682
Black–Scholes call price (Q3): 75.65999977609337
Difference (binomial - BS): 0.032973130503449966
